# 06 - Gold Pipeline Municipal

Evidencia reproducible de la capa Gold para presupuesto y ejecucion de ingresos municipales.

```mermaid
flowchart LR
    S["Silver curado"] --> G["Gold analitico"]
    G --> D["Dimensiones"]
    G --> F["Facts y marts"]
    D --> P["Power BI: 6 dashboards"]
    F --> P
```


In [1]:
import json
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName('gold-notebook-evidence').getOrCreate()
root = Path('/home/jovyan/work')
gold = root / 'data' / 'gold'
snapshots = sorted((root / 'data' / 'audit' / 'metrics').rglob('gold_summary_*.json'))
summary = json.loads(snapshots[-1].read_text(encoding='utf-8'))
assert summary['published_tables'] == 11 and not summary['errors']
print('Snapshot:', snapshots[-1].name)
print('Tablas:', summary['published_tables'], 'Registros:', summary['records_published'])
print('Checks fallidos:', summary['failed_quality_checks'])


Snapshot: gold_summary_20260601_175502.json
Tablas: 11 Registros: 9486679
Checks fallidos: 0


## Inventario Gold

El pipeline publica cinco dimensiones y seis facts o marts en Parquet Snappy.

In [2]:
inventory = [(item['table_name'], item['records_published']) for item in summary['published']]
spark.createDataFrame(inventory, ['tabla', 'filas']).orderBy('tabla').show(20, truncate=False)


+--------------------------------+-------+
|tabla                           |filas  |
+--------------------------------+-------+
|dim_clasificador_ingreso        |2783   |
|dim_formulario_sismepre         |94     |
|dim_municipalidad_gold          |1964   |
|dim_pregunta_sismepre           |696    |
|dim_tiempo                      |172    |
|fact_calidad_datos              |3239   |
|fact_ingresos_clasificador      |8880634|
|fact_ingresos_mensuales         |323159 |
|fact_predial_mensual            |49368  |
|fact_sismepre_cumplimiento      |19037  |
|fact_sismepre_respuestas_resumen|205533 |
+--------------------------------+-------+



## Cobertura Municipal

RENAMU se usa como enriquecimiento parcial. SISMEPRE tampoco funciona como filtro: las entidades solo SIAF siguen disponibles para el analisis.

El alcance final de la exposicion se controla con `data/reference/municipalidades_presentadas.csv`. Gold agrega `in_scope_presentacion`; si la plantilla esta vacia, el pipeline registra `pendiente_archivo_profesor` y conserva todas las municipalidades.

Tratamiento temporal: SIAF es serie mensual 2012-2026; SISMEPRE usa los anios y periodos disponibles; RENAMU 2022 se trata como fuente estatica de enriquecimiento.

No se usan mapas externos porque las tres fuentes no traen latitud, longitud ni geometria. El analisis territorial se presenta con rankings, tablas y segmentadores por UBIGEO, departamento, provincia y distrito.

Categorias municipales: `CategoriasMunicipalidades.csv` se trata como archivo de referencia del profesor. Como no trae `SEC_EJEC` ni `UBIGEO`, Gold cruza por nombre normalizado y no asigna categoria cuando el match es ambiguo. Los campos publicados son `categoria_municipalidad` y `categoria_match_status`.

In [3]:
municipios = spark.read.parquet(str(gold / 'dim_municipalidad_gold'))
coverage = municipios.agg(
    F.count('*').alias('municipalidades_siaf'),
    F.sum(F.col('has_sismepre').cast('int')).alias('con_sismepre'),
    F.sum((~F.col('has_sismepre')).cast('int')).alias('solo_siaf'),
    F.sum(F.col('renamu_match').cast('int')).alias('match_renamu'),
).first().asDict()
assert coverage == {'municipalidades_siaf': 1964, 'con_sismepre': 1111, 'solo_siaf': 853, 'match_renamu': 587}
coverage


{'municipalidades_siaf': 1964,
 'con_sismepre': 1111,
 'solo_siaf': 853,
 'match_renamu': 587}

## Esquemas Y Calidad

Las facts conservan trazabilidad Silver y agregan metadatos Gold. La auditoria aplica completitud, unicidad, validez, consistencia, integridad, actualidad, disponibilidad y exactitud.

In [4]:
ingresos = spark.read.parquet(str(gold / 'fact_ingresos_mensuales'))
ingresos.printSchema()
quality = spark.read.parquet(str(gold / 'fact_calidad_datos'))
quality.groupBy('layer', 'status').count().orderBy('layer', 'status').show(truncate=False)


root
 |-- SEC_EJEC: string (nullable = true)
 |-- ANO_DOC: integer (nullable = true)
 |-- MES_DOC: integer (nullable = true)
 |-- MONTO_PIA: decimal(24,2) (nullable = true)
 |-- MONTO_PIM: decimal(24,2) (nullable = true)
 |-- MONTO_RECAUDADO: decimal(24,2) (nullable = true)
 |-- _gold_source_row_count: long (nullable = true)
 |-- _bronze_dataset: string (nullable = true)
 |-- _bronze_asset_name: string (nullable = true)
 |-- _bronze_table_name: string (nullable = true)
 |-- _bronze_asset_role: string (nullable = true)
 |-- _bronze_source_type: string (nullable = true)
 |-- _bronze_source_path: string (nullable = true)
 |-- _bronze_source_url: string (nullable = true)
 |-- _bronze_source_checksum: string (nullable = true)
 |-- _bronze_execution_id: string (nullable = true)
 |-- _bronze_ingestion_ts: timestamp (nullable = true)
 |-- _bronze_ingestion_date: date (nullable = true)
 |-- _silver_source_row_count: long (nullable = true)
 |-- _silver_execution_id: string (nullable = true)
 |--

## KPIs De Ejemplo

La fact mensual permite calcular directamente presupuesto, recaudacion, variacion y avance de ejecucion.

In [5]:
latest_year = ingresos.agg(F.max('ANO_DOC')).first()[0]
kpi = ingresos.filter(F.col('ANO_DOC') == latest_year).agg(
    F.sum('MONTO_PIA').alias('pia'),
    F.sum('MONTO_PIM').alias('pim'),
    F.sum('MONTO_RECAUDADO').alias('recaudado'),
).withColumn('pct_ejecucion', F.when(F.col('pim') != 0, F.round(F.col('recaudado') / F.col('pim') * 100, 2)))
print('Anio mas reciente:', latest_year)
kpi.show(truncate=False)


Anio mas reciente: 2026
+--------------+--------------+--------------+-------------+
|pia           |pim           |recaudado     |pct_ejecucion|
+--------------+--------------+--------------+-------------+
|30394734607.00|39534088707.00|18411947189.28|46.57        |
+--------------+--------------+--------------+-------------+



## Mapeo Para Los Seis Dashboards

| Dashboard | Tablas Gold |
|---|---|
| Evolucion mensual del presupuesto y recaudacion | `fact_ingresos_mensuales`, `dim_tiempo` |
| Avance de ejecucion por municipalidad | `fact_ingresos_mensuales`, `dim_municipalidad_gold` |
| Ranking territorial por recaudacion | `fact_ingresos_mensuales`, `dim_municipalidad_gold` |
| Indicadores de impuesto predial | `fact_predial_mensual`, `dim_municipalidad_gold` |
| Cobertura y cumplimiento SISMEPRE | `fact_sismepre_cumplimiento`, `fact_sismepre_respuestas_resumen` |
| Calidad y cobertura de datos | `fact_calidad_datos`, `dim_municipalidad_gold` |
